In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

C:\Users\dell\AppData\Local\Temp\ipykernel_21980\4208644247.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database, so I can’t see the exact count. If you share a bit more about your setup, I can give you the exact query. In the meantime, here are common options:

- If you want the total number of artist records:
  - SQL (generic): SELECT COUNT(*) AS total_artists FROM artists;
  - If your table is named differently, replace artists with the actual table name.

- If you want to count distinct artists by ID (to avoid duplicates):
  - SELECT COUNT(DISTINCT artist_id) AS total_artists FROM artists;

- If you want to count only active artists (assuming an is_active flag):
  - SELECT COUNT(*) AS total_artists FROM artists WHERE is_active = 1;

- If you’re using a data warehouse with a dimensional table (e.g., dim_artists):
  - SELECT COUNT(*) AS total_artists FROM dim_artists WHERE is_active = 1;

Tell me:
- the database type (PostgreSQL, MySQL/MariaDB, SQL Server, SQLite, etc.),
- the table name that stores artists,
- whether you want all records or only active/dist

In [8]:
from langchain.messages import HumanMessage
config=  {"configurable": {"thread_id": "abcd1234"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    config=config,
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

275

Note: The table in this database is named Artist (singular, case-sensitive in SQLite). The initial query using "artists" didn't exist, which caused the error. The correct query SELECT COUNT(*) FROM Artist returns 275. If you want, I can run additional checks or fetch sample artist names.


In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="yes please give me top 10 artist names.")]},
    config=config,
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)



Sure—what do you mean by “top”?

- Global current popularity (e.g., top 10 artists by Spotify monthly listeners)
- Daily/weekly trending artists (Spotify charts today)
- Top 10 artists by genre (pop, rock, hip-hop, K-pop, etc.)
- Top 10 artists by region (US, UK, etc.)
- All-time influential/top-ranked artists (historical list)

If you don’t have a preference, I can pull a current global list by Spotify monthly listeners as a default. Want me to do that and cite the source?


In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Top 10 artists by genre (pop, rock, hip-hop, K-pop, etc.)")]},
    config=config,
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

I can do that, but I need a couple of clarifications to pull the right data:

- Timeframe: current week (or latest chart) or all-time/top historically?
- Geography: global, or a specific region (e.g., US)?
- Genres: do you want these core genres only (Pop, Rock, Hip-Hop/Rap, K-Pop, EDM, R&B, Country, Latin, Alternative, Metal), or a different list?
- Data source: would you prefer I base this on current Spotify/YouTube chart data, Billboard charts, or a blended/summary from multiple sources?

If you’d like, I can start with a default setup: global, current week, top 10 per genre for Pop, Rock, Hip-Hop/Rap, K-Pop, EDM, R&B, Country, Latin, Alternative, and Metal, using a blended data approach and noting sources. Would you like me to proceed with that?
